<!--
    Copyright (C) 2026 The AiQuokkaWannabe Authors <https://github.com/ai-quokka-wannabe/iso-lab>

    This program is free software: you can redistribute it and/or modify it under the terms of
    the GNU General Public License as published by the Free Software Foundation, either version
    3 of the License, or (at your option) any later version.

    This program is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY;
    without even the implied warranty of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.
    See the GNU General Public License for more details.

    You should have received a copy of the GNU General Public License along with this program.
    If not, see <https://www.gnu.org/licenses/>.
-->

# The Midbody Speed With the Published Choices, Against the Dataset's Own

The check written down in the journal of 2026-10-15, *The Midbody Speed With the Published Choices, at Eight Arcs*,
before it ran: on the seventeen block-1 records of 2026-10-14, all counted, the eight-arc midbody speed with the
published speed's choices (the Rust binary `midbody-speed --published-choices`) passes if in every record its Pearson
correlation with the published `midbody_speed` is at least 0.95 and the slope of a least-squares line through the
origin, the lab's on the published, lies between 0.9 and 1.1, over the frames where both exist. The measure of
2026-10-14 (`midbody-speed`) is run beside it under the same bar for comparison. Sign agreement and the median
absolute difference are reported, not tested.

Run from the repository's root with the block-1 files in `data/raw/owmd-n2/` (fetched by `fetch-block`, which names
a file only once its MD5 matches) and the Rust toolchain installed; the notebook builds and calls the binary.


In [ ]:
import os, subprocess
import numpy as np
import h5py

os.chdir(os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd())
DIRECTORY, SUFFIX = "data/raw/owmd-n2", "_features.hdf5"
# The seventeen records of the check of 2026-10-14, pinned so that a re-run checks the same set.
records = ["1006562", "1007123", "1007405", "1008698", "1011246", "1011901", "1013148", "1014831", "1020507",
           "1021835", "1024910", "1025254", "1025397", "1029332", "1030984", "1031252", "1032937"]
missing = [record for record in records if not os.path.exists(f"{DIRECTORY}/{record}{SUFFIX}")]
assert not missing, f"fetch these first: {missing}"
subprocess.run(["cargo", "build", "--quiet", "--release", "--bin", "midbody-speed"], check=True)
print(len(records), "records checked")
print("h5py", h5py.__version__, "numpy", np.__version__)

In [ ]:
def lab_speeds(record, published_choices):
    flags = ["--published-choices"] if published_choices else []
    out = subprocess.run(["target/release/midbody-speed", *flags, record], check=True, capture_output=True,
                         text=True).stdout
    return np.array([float(line.split("\t")[1]) for line in out.splitlines()[1:]])

def published_speeds(record):
    with h5py.File(f"{DIRECTORY}/{record}{SUFFIX}", "r") as file:
        return file["/features_timeseries"]["midbody_speed"].astype(float)

def compare(ours, theirs):
    assert len(ours) == len(theirs), (len(ours), len(theirs))
    both = ~np.isnan(ours) & ~np.isnan(theirs)
    a, b = ours[both], theirs[both]
    r = np.corrcoef(a, b)[0, 1]
    slope = np.sum(a * b) / np.sum(b * b)
    return (both.sum(), r, slope, np.mean(np.sign(a) == np.sign(b)), np.median(np.abs(a - b)),
            r >= 0.95 and 0.9 <= slope <= 1.1)

rows = {True: [], False: []}
for record in records:
    theirs = published_speeds(record)
    for choices in (True, False):
        rows[choices].append((record, *compare(lab_speeds(record, choices), theirs)))
for choices, title in ((True, "published choices (the check)"), (False, "the measure of 2026-10-14 (comparison)")):
    print(title)
    print("record   frames  r      slope  sign   median|diff| um/s  passes")
    for record, n, r, slope, sign, diff, ok in rows[choices]:
        print(f"{record}  {n:6d}  {r:.3f}  {slope:.3f}  {sign:.3f}  {diff:6.2f}             {ok}")
    print("every record passes:", all(row[-1] for row in rows[choices]))
    print("medians over records: r %.3f, slope %.3f, sign %.3f, |diff| %.2f" % tuple(
        np.median([row[i] for row in rows[choices]]) for i in (2, 3, 4, 5)))
    print()
higher = sum(new[2] > old[2] for new, old in zip(rows[True], rows[False]))
print(f"correlation higher with the published choices in {higher} of {len(records)} records")